# DermaLens / DermSynReal — Acne SD 2.1 LoRA (Kaggle T4)
**Version 7 — HF repository corrected and audited | controlled, resumable UNet-LoRA training | one cell at a time**

- Uses **train/Acne only** for learning. Validation/test remain untouched (the script's `validation_prompt` creates *synthetic* monitoring images, not held-out validation data).
- Uses Hugging Face Diffusers **official `examples/text_to_image/train_text_to_image_lora.py`**, fixed to `v0.35.1` for reproducibility. No custom diffusion loss implementation.
- Every training invocation gets its own timestamped run folder with `command.json`, `environment.json`, `train.log`, `gpu_usage.csv`, and TensorBoard event files. Training checkpoints additionally include optimizer/accelerator state.
- **Internet ON, GPU T4 selected** in Kaggle Settings. You may need Hugging Face access to the base-model weights; do not paste tokens into notebook cells.
- **Smoke test first (10 optimizer steps)**, inspect outputs, then change `RUN_MODE` to `full` for 500 steps. This is not proof of diagnostic accuracy.
- **Important:** 512×512 default center crop may exclude facial lesions on some portraits. Preflight contact sheets and metadata checks are required; center crop here is a simple default, not a medical segmentation solution.

Official source: https://github.com/huggingface/diffusers/tree/v0.35.1/examples/text_to_image

- **Model provenance:** Uses a community-hosted SD2.1 Base repository. Log the resolved repository revision before a research run; model equivalence to the removed official repository has not been independently established.


## 1 — Kaggle runtime and settings

In [ ]:
import os, sys, subprocess, json, platform, time
from pathlib import Path
print('Python:',sys.version)
print('CUDA GPU:')
subprocess.run(['nvidia-smi'], check=True)
assert Path('/kaggle/input').exists(), 'This notebook is designed to run in Kaggle.'
print('Data sources:', [x.name for x in Path('/kaggle/input').iterdir()])

## 2 — Install a mutually compatible package set
Do this once, **before imports of Diffusers**. On a fresh Kaggle environment, restart the kernel if pip asks you to; rerun cells afterward. PyTorch/CUDA are supplied by Kaggle and are **not** replaced.

In [ ]:
import subprocess, sys
packages = [
    'diffusers==0.35.1', 'transformers==4.56.2', 'accelerate==1.10.1',
    'peft==0.17.1', 'datasets==3.6.0', 'huggingface_hub==0.34.4',
    'safetensors>=0.4.5,<1', 'tensorboard>=2.15,<3', 'ftfy', 'sentencepiece'
]
subprocess.run([sys.executable,'-m','pip','install','-q',*packages], check=True)
print('Install complete. If imports fail after an environment upgrade, restart session and rerun.')

## 3 — Version + GPU compatibility checks
Fail clearly instead of training against an unexpected version.

In [ ]:
import torch, diffusers, transformers, accelerate, peft, datasets, huggingface_hub
from packaging.version import Version
versions={x:getattr(sys.modules[x],'__version__','unknown') for x in ['torch','diffusers','transformers','accelerate','peft','datasets','huggingface_hub']}
print(json.dumps(versions,indent=2))
assert diffusers.__version__ == '0.35.1', 'Restart kernel; wrong Diffusers version loaded'
assert torch.cuda.is_available(), 'Enable Kaggle T4 GPU under Settings -> Accelerator'
print('CUDA:',torch.version.cuda,'GPU:',torch.cuda.get_device_name(0),'VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1e9,2))
assert torch.cuda.get_device_properties(0).total_memory >= 13e9, 'This configuration targets ~16GB T4; inspect GPU type.' 

## 3B — Optional Hugging Face access through Kaggle Secrets
If you have saved a token in Kaggle Secrets as `HF_TOKEN`, this cell uses it without printing the token. Otherwise, public model access is attempted normally. Never paste tokens into code or publish them in outputs.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    secret=UserSecretsClient().get_secret('HF_TOKEN')
    if secret:
        os.environ['HF_TOKEN']=secret
        print('HF_TOKEN secret detected (value hidden).')
    else:
        print('No HF_TOKEN secret configured; using anonymous model access.')
except Exception as e:
    print('HF_TOKEN secret unavailable; anonymous model access will be attempted:',type(e).__name__)


## 4 — Configuration
Set `RUN_MODE = "smoke"` initially. Later set it to `"full"`. Full and smoke outputs never overwrite each other. For resuming a previous interrupted run, put its **absolute output directory** in `RESUME_RUN_DIR` and set `RUN_MODE="full"`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
RUN_MODE = 'smoke'              # smoke | full
RESUME_RUN_DIR = ''              # e.g. /kaggle/working/acne_lora_runs/full_20261030T120000Z/output
MODEL_ID = 'sd2-community/stable-diffusion-2-1-base'
SEED = 42
RESOLUTION = 512
RANK = 16
LEARNING_RATE = 1e-4
TRAIN_BATCH_SIZE = 1
GRAD_ACCUM = 4
SMOKE_STEPS = 10
FULL_STEPS = 500
CHECKPOINT_EVERY = 100       # for full run
VALIDATION_PROMPT = 'Clinical facial photograph, a human face with visible acne, realistic natural skin texture, neutral lighting'
OUTPUT_ROOT = Path('/kaggle/working/acne_lora_runs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
assert RUN_MODE in ('smoke','full')
if RESUME_RUN_DIR:
    assert RUN_MODE == 'full', 'Resume is only configured for full training'
    OUTPUT_DIR = Path(RESUME_RUN_DIR).resolve()
    assert OUTPUT_DIR.is_dir(), 'Resume directory not found; restore it to /kaggle/working first'
    RUN_DIR = OUTPUT_DIR.parent
    assert (RUN_DIR/'command.json').is_file(), 'Cannot resume: missing original command.json'
    assert any(OUTPUT_DIR.glob('checkpoint-*')), 'Cannot resume: no checkpoint directory found'
else:
    timestamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    RUN_DIR=OUTPUT_ROOT/f'{RUN_MODE}_{timestamp}_{os.getpid()}'
    RUN_DIR.mkdir(parents=True,exist_ok=False)
    OUTPUT_DIR=RUN_DIR/'output'
OUTPUT_DIR.mkdir(exist_ok=True)
print('Run:',RUN_DIR,' -- Output:',OUTPUT_DIR)

## 5 — Find the dataset and audit image files
Do **not** mix the other classes or validation/test images into training.
Image integrity, dimensions and counts are recorded in `input_audit.json`. Invalid/corrupt images stop training. The Kaggle input dataset is never modified.


In [ ]:
from PIL import Image, ImageOps, UnidentifiedImageError
from collections import Counter
import csv
roots=[p for p in Path('/kaggle/input').rglob('train/Acne') if p.is_dir()]
assert len(roots)==1, f'Expected one train/Acne folder; found {roots}. Check dataset mount.'
ACNE_DIR=roots[0]
DATASET_ROOT=ACNE_DIR.parent.parent
print('Dataset root:',DATASET_ROOT)
print('Root entries:',sorted(p.name for p in DATASET_ROOT.iterdir()))
assert (DATASET_ROOT/'validation'/'Acne').is_dir() and (DATASET_ROOT/'test'/'Acne').is_dir(), 'Expected validation and test Acne folders'
EXT={'.png','.jpg','.jpeg','.webp','.bmp'}
files=sorted(p for p in ACNE_DIR.rglob('*') if p.is_file() and p.suffix.lower() in EXT)
print('Candidate Acne training images:',len(files)); assert len(files)>=20,'Too few images; check input dataset'
records=[]; valid=[]; invalid=[]
for idx,p in enumerate(files):
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            im=ImageOps.exif_transpose(im)
            w,h=im.size
            assert w>=128 and h>=128, 'Image smaller than 128px'
            assert im.getbbox() is not None, 'Blank image'
        valid.append(p)
        records.append({'file':str(p.relative_to(DATASET_ROOT)),'width':w,'height':h})
    except Exception as e: invalid.append({'file':str(p),'error':repr(e)})
with (RUN_DIR/'input_audit.json').open('w') as f:
    json.dump({'count':len(files),'valid':len(valid),'invalid':invalid,'records':records},f,indent=2)
print('valid:',len(valid),'invalid:',len(invalid))
print('Sample shapes:',Counter((r['width'],r['height']) for r in records).most_common(8))
assert not invalid, 'Invalid images detected. See input_audit.json before training'
assert len(valid)>0

## 6 — Show original images and assess cropping
No image augmentation here. Check whether facial areas/lesions remain visible if resized and centrally cropped to 512×512.

In [ ]:
import random, matplotlib.pyplot as plt
from PIL import ImageOps
sample=random.Random(SEED).sample(valid,min(8,len(valid)))
fig,axes=plt.subplots(2,4,figsize=(16,8))
for ax,p in zip(axes.flat,sample):
    with Image.open(p) as im:
        display_im=ImageOps.fit(ImageOps.exif_transpose(im).convert('RGB'),(512,512),method=Image.Resampling.LANCZOS,centering=(.5,.5))
        ax.imshow(display_im)
    ax.set_title(p.name[:22],fontsize=9);ax.axis('off')
for ax in list(axes.flat)[len(sample):]: ax.axis('off')
plt.tight_layout();plt.savefig(RUN_DIR/'crop_contact_sheet.png',dpi=130);plt.show()
print('ACTION: Inspect the contact sheet before training. If lesions are cropped off, stop and adjust image preparation.')

## 7 — Build captioned training dataset
The official script expects an imagefolder containing `metadata.jsonl` with `file_name` and `text`. **Generic captions are a temporary baseline**, not clinical annotations. They do NOT invent subtype or severity. For a proper research run, use reviewed image-specific captions. This cell never reads validation/test images.

In [ ]:
import shutil, json
STAGING=RUN_DIR/'captioned_acne_train'
caption='Clinical photograph of a human face with acne, natural skin texture, dermatology research image'
if RESUME_RUN_DIR:
    assert (STAGING/'metadata.jsonl').is_file(), 'Cannot resume: missing original metadata.jsonl in staging directory'
    metadata=[json.loads(line) for line in (STAGING/'metadata.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
    assert metadata and all((STAGING/m['file_name']).is_file() for m in metadata), 'Cannot resume: staged images missing'
    original=json.loads((RUN_DIR/'command.json').read_text())
    cfg=original['config']
    assert cfg['rank']==RANK and cfg['lr']==LEARNING_RATE and cfg['resolution']==RESOLUTION and cfg['accumulation']==GRAD_ACCUM, 'Resume hyperparameters differ from original run'
    assert original['train_count']==len(metadata), 'Resume training image count differs'
    assert original['model_id']==MODEL_ID, 'Resume base model differs'
    print('Resuming from original captioned data; image count:',len(metadata))
else:
    STAGING.mkdir(exist_ok=False)
    metadata=[]
    for i,p in enumerate(valid):
        target=f'image_{i:06d}{p.suffix.lower()}'
        shutil.copy2(p,STAGING/target)
        metadata.append({'file_name':target,'text':caption})
    with (STAGING/'metadata.jsonl').open('w',encoding='utf-8') as f:
        for item in metadata: f.write(json.dumps(item,ensure_ascii=False)+'\n')
    assert len(metadata)==len(valid)
    assert all((STAGING/m['file_name']).is_file() for m in metadata)
    print('Prepared:',len(metadata),'captioned training images at',STAGING)
print('Example:',metadata[0]);print('For final research results, replace generic captions with reviewed image-specific captions.')

## 7B — Preflight the imagefolder metadata (before GPU training)
Confirm the official dataset loader recognizes both `image` and `text` fields and can decode an actual example. This avoids spending GPU time discovering a dataset schema issue.

In [ ]:
from datasets import load_dataset
from PIL import Image
import importlib
required=['torch','torchvision','diffusers','transformers','accelerate','peft','datasets','safetensors','tensorboard','ftfy','sentencepiece']
for name in required:
    mod=importlib.import_module(name)
    print('Import OK:',name,getattr(mod,'__version__',''))
preview_ds=load_dataset('imagefolder',data_files={'train':str(STAGING/'**')},split='train')
print('Dataset columns:',preview_ds.column_names,'Rows:',len(preview_ds))
assert 'image' in preview_ds.column_names and 'text' in preview_ds.column_names
assert len(preview_ds)==len(metadata), 'Captioned dataset has unexpected number of rows'
sample_row=preview_ds[0]
assert isinstance(sample_row['image'],Image.Image) and isinstance(sample_row['text'],str)
assert sample_row['text'].strip()
print('Decoded sample:',sample_row['image'].size,'caption:',sample_row['text'][:100])
del preview_ds, sample_row


## 8 — Obtain exact tagged official training script
This step needs Kaggle Internet ON. We pin the script and library to Diffusers `v0.35.1`, then record SHA256 of the script.

In [ ]:
import subprocess, sys, hashlib
from pathlib import Path
REPO=Path('/kaggle/working/diffusers_examples_v0351')
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--depth','1','--branch','v0.35.1','https://github.com/huggingface/diffusers.git',str(REPO)],check=True)
else:
    tag=subprocess.check_output(['git','-C',str(REPO),'describe','--tags','--exact-match'],text=True).strip()
    assert tag=='v0.35.1',f'Unexpected checked out tag: {tag}'
SCRIPT=REPO/'examples/text_to_image/train_text_to_image_lora.py'
assert SCRIPT.exists()
script_sha=hashlib.sha256(SCRIPT.read_bytes()).hexdigest()
print('Script:',SCRIPT,'SHA256:',script_sha)
subprocess.run([sys.executable,str(SCRIPT),'--help'],check=True,stdout=subprocess.DEVNULL)
print('Official training script parsed successfully')

## 9 — Preflight base model (weights, tokenizer, scheduler)
The SD 2.1 checkpoint is downloaded through Hugging Face. Access terms and model availability can change; report a download/authentication error rather than silently substituting another model.

In [ ]:
import json
from pathlib import Path
from huggingface_hub import hf_hub_download
from diffusers import DDPMScheduler

# Check small config files first, so invalid repo IDs fail before large downloads.
required_configs = [
    'model_index.json',
    'scheduler/scheduler_config.json',
    'unet/config.json',
    'vae/config.json',
    'text_encoder/config.json',
    'tokenizer/tokenizer_config.json',
]
try:
    config_paths = {name: hf_hub_download(repo_id=MODEL_ID, filename=name) for name in required_configs}
except Exception as e:
    raise RuntimeError(
        f'Unable to read required model files from {MODEL_ID}. '
        'Check Kaggle Internet, the Hugging Face model page, and token permissions if needed. '
        f'Original error: {e}'
    ) from e
scheduler_config = json.loads(Path(config_paths['scheduler/scheduler_config.json']).read_text())
print('Verified model repository:', MODEL_ID)
for name in config_paths: print('  OK:', name)
print('Scheduler prediction_type:', scheduler_config.get('prediction_type'))
assert scheduler_config.get('prediction_type') in ('v_prediction', 'epsilon'), 'Unexpected scheduler prediction type'

from huggingface_hub import model_info
model_revision = model_info(MODEL_ID).sha
(RUN_DIR/'base_model_revision.txt').write_text(f'{MODEL_ID}@{model_revision}\n')
print('Base model commit:',model_revision)


## 10 — Record run provenance and execute training
**Warnings and errors from training are captured in `train.log`; GPU utilization is sampled in `gpu_usage.csv`.** Checkpoints are written into `output/checkpoint-STEP`. If a run errors, the log and saved state remain for diagnosis. Do not re-execute this cell without intentionally choosing a new output folder or resume directory.

In [ ]:
import threading, subprocess, shlex, time, traceback, os, csv, json, sys, platform
from datetime import datetime, timezone
STEPS=SMOKE_STEPS if RUN_MODE=='smoke' else FULL_STEPS
ckpt_interval=5 if RUN_MODE=='smoke' else CHECKPOINT_EVERY
cmd=[sys.executable,'-m','accelerate.commands.launch','--num_processes=1','--mixed_precision=fp16',
     str(SCRIPT),'--pretrained_model_name_or_path',MODEL_ID,
     '--train_data_dir',str(STAGING), '--image_column','image','--caption_column','text',
     '--resolution',str(RESOLUTION),'--center_crop',
     '--train_batch_size',str(TRAIN_BATCH_SIZE),'--gradient_accumulation_steps',str(GRAD_ACCUM),
     '--max_train_steps',str(STEPS),'--learning_rate',str(LEARNING_RATE),
     '--lr_scheduler','constant','--lr_warmup_steps','0','--gradient_checkpointing',
     '--mixed_precision','fp16','--rank',str(RANK),
     '--checkpointing_steps',str(ckpt_interval),'--checkpoints_total_limit','3',
     '--seed',str(SEED),'--output_dir',str(OUTPUT_DIR),'--report_to','tensorboard']
# Do not trigger official script's per-epoch image generation while doing T4 training;
# we do controlled post-training validation in a separate cell.
if RESUME_RUN_DIR: cmd += ['--resume_from_checkpoint','latest']
provenance={'run_mode':RUN_MODE,'model_id':MODEL_ID,'config':{'rank':RANK,'steps':STEPS,'lr':LEARNING_RATE,'resolution':RESOLUTION,'accumulation':GRAD_ACCUM},
            'dataset_root':str(DATASET_ROOT),'train_count':len(valid),'script_sha256':script_sha,'command':cmd,'versions':versions,
            'timestamp_utc':datetime.now(timezone.utc).isoformat()}
if not RESUME_RUN_DIR:
    (RUN_DIR/'command.json').write_text(json.dumps(provenance,indent=2))
else:
    (RUN_DIR/'resume_command.json').write_text(json.dumps(provenance,indent=2))
(RUN_DIR/'environment.json').write_text(json.dumps({'python':sys.version,'platform':platform.platform(),'torch_cuda':torch.version.cuda,'versions':versions},indent=2))
(RUN_DIR/'train_command.sh').write_text('#!/bin/bash\n'+' '.join(shlex.quote(x) for x in cmd)+'\n')
stop_monitor=threading.Event()
def gpu_monitor():
    with (RUN_DIR/'gpu_usage.csv').open('a',newline='') as f:
        w=csv.writer(f); w.writerow(['timestamp_utc','utilization_gpu_pct','memory_used_MiB','memory_total_MiB'])
        while not stop_monitor.is_set():
            try:
                raw=subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader,nounits'],text=True,timeout=8).strip()
                for line in raw.splitlines(): w.writerow([datetime.now(timezone.utc).isoformat(),*[x.strip() for x in line.split(',')]])
                f.flush()
            except Exception as e: print('GPU monitoring warning:',e)
            stop_monitor.wait(15)
monitor=threading.Thread(target=gpu_monitor,daemon=True);monitor.start()
returncode=None
try:
    with (RUN_DIR/'train.log').open('a',encoding='utf-8') as log:
        log.write('\n--- START '+datetime.now(timezone.utc).isoformat()+' ---\n');log.flush()
        proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env={**os.environ,'TOKENIZERS_PARALLELISM':'false'})
        try:
            for line in proc.stdout:
                print(line,end=''); log.write(line); log.flush()
            returncode=proc.wait()
        except KeyboardInterrupt:
            proc.terminate(); proc.wait(timeout=20); raise
        finally:
            log.write('\n--- EXIT '+str(returncode)+' '+datetime.now(timezone.utc).isoformat()+' ---\n')
finally:
    stop_monitor.set();monitor.join(timeout=20)
(RUN_DIR/'run_status.json').write_text(json.dumps({'returncode':returncode,'finished_utc':datetime.now(timezone.utc).isoformat()}))
assert returncode==0,f'Training failed with exit code {returncode}; inspect {RUN_DIR}/train.log'
print('Training finished. Checkpoint files:',list(OUTPUT_DIR.glob('*.safetensors')))

## 11 — Inspect artifacts and plot scalar losses
Official Diffusers uses TensorBoard `step_loss`, so this reads event files where available. If no records are present, refer to `train.log` and send us the error.

In [ ]:
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
logs=list(OUTPUT_DIR.rglob('events.out.tfevents.*'))
print('TensorBoard event files:',[str(x) for x in logs])
points=[]
for logfile in logs:
    try:
        acc=EventAccumulator(str(logfile));acc.Reload()
        print('Tags:',acc.Tags().get('scalars',[]))
        for tag in acc.Tags().get('scalars',[]):
            if 'loss' in tag.lower():
                points += [(e.step,e.value,tag) for e in acc.Scalars(tag)]
    except Exception as e: print('Event parse warning:',e)
if points:
    fig,ax=plt.subplots(figsize=(9,4))
    for tag in sorted({x[2] for x in points}):
        pts=sorted((s,v) for s,v,t in points if t==tag)
        ax.plot([p[0] for p in pts],[p[1] for p in pts],label=tag)
    ax.set(xlabel='Optimizer step',ylabel='Recorded training loss',title='Acne LoRA training log')
    ax.legend();plt.tight_layout();plt.savefig(RUN_DIR/'training_loss.png',dpi=150);plt.show()
print('Output safetensors:',[str(p) for p in OUTPUT_DIR.glob('*.safetensors')]);print('Saved checkpoints:',[p.name for p in OUTPUT_DIR.glob('checkpoint-*')])

## 12 — Reload the exported LoRA and generate a fixed-seed smoke grid
**Separate GPU-memory stage:** unload training components first if necessary. Generated images are for technical review, **not for clinical use or diagnosis**. Compare with the unadapted model later using the same prompts and seeds.

In [ ]:
import gc, torch
import matplotlib.pyplot as plt
from diffusers import StableDiffusionPipeline
adapter=OUTPUT_DIR/'pytorch_lora_weights.safetensors'
assert adapter.exists(),f'Final adapter not found: {adapter}. Training must finish successfully first.'
gc.collect();torch.cuda.empty_cache()
pipe=StableDiffusionPipeline.from_pretrained(MODEL_ID,torch_dtype=torch.float16,safety_checker=None,requires_safety_checker=False)
pipe.load_lora_weights(str(OUTPUT_DIR))
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe=pipe.to('cuda')
prompts=[
    'Clinical facial photograph of a human face with acne on both cheeks, realistic skin texture, neutral background',
    'Clinical facial photograph of a human face with visible acne on the forehead, realistic texture, neutral lighting',
    'Clinical facial photograph of a human face with acne along the jawline, natural skin texture, neutral lighting',
    'Clinical facial photograph, frontal view, human face with visible acne, realistic dermatology image'
]
imgs=[]
for j,prompt in enumerate(prompts):
    result=pipe(prompt,num_inference_steps=25,guidance_scale=7.0,generator=torch.Generator(device='cuda').manual_seed(SEED+j))
    im=result.images[0];im.save(RUN_DIR/f'generated_{j:02d}.png');imgs.append(im)
fig,axes=plt.subplots(1,4,figsize=(16,4))
for ax,im in zip(axes,imgs): ax.imshow(im);ax.axis('off')
plt.tight_layout();plt.savefig(RUN_DIR/'generation_grid.png',dpi=130);plt.show()
del pipe;gc.collect();torch.cuda.empty_cache()
print('Generated samples and grid saved in',RUN_DIR)

## 13 — Export standalone adapter for the synthetic-generation inference notebook

This creates a small inference bundle with weights, base model ID, SHA256, and a loader example. The full checkpoint archive is separate.


In [ ]:
import json, shutil, hashlib, zipfile
EXPORT_DIR=RUN_DIR/'inference_bundle'
EXPORT_DIR.mkdir(exist_ok=True)
weights=OUTPUT_DIR/'pytorch_lora_weights.safetensors'
assert weights.is_file() and weights.stat().st_size>1024, 'Final LoRA weight file missing'
shutil.copy2(weights,EXPORT_DIR/weights.name)
manifest={'class':'Acne','adapter_type':'diffusers_unet_lora','base_model':MODEL_ID,'base_model_revision':model_revision,'prediction_type':scheduler_config.get('prediction_type'),'rank':RANK,'resolution':RESOLUTION,'steps':STEPS,'run_mode':RUN_MODE,'weight_file':weights.name,'sha256':hashlib.sha256(weights.read_bytes()).hexdigest(),'diffusers_version':diffusers.__version__,'research_only':True}
(EXPORT_DIR/'adapter_manifest.json').write_text(json.dumps(manifest,indent=2))
example = """# Extract the inference bundle and update adapter_dir to its Kaggle input location.
import torch
from diffusers import StableDiffusionPipeline
adapter_dir = '/kaggle/input/YOUR_ADAPTER_DATASET/inference_bundle'
pipe = StableDiffusionPipeline.from_pretrained('sd2-community/stable-diffusion-2-1-base', torch_dtype=torch.float16)
pipe.load_lora_weights(adapter_dir, weight_name='pytorch_lora_weights.safetensors')
pipe.to('cuda')
img=pipe('Clinical facial photograph with visible acne, natural skin texture',num_inference_steps=25,generator=torch.Generator('cuda').manual_seed(42)).images[0]
img.save('/kaggle/working/acne_example.png')
"""
(EXPORT_DIR/'inference_example.py').write_text(example)
EXPORT_ZIP=Path('/kaggle/working')/f'{RUN_DIR.name}_INFERENCE_BUNDLE.zip'
with zipfile.ZipFile(EXPORT_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for p in EXPORT_DIR.iterdir():
        if p.is_file(): z.write(p,arcname='inference_bundle/'+p.name)
print('Reusable adapter bundle:',EXPORT_ZIP, 'bytes:',EXPORT_ZIP.stat().st_size)


## 14 — Archive the complete run
Kaggle working files are ephemeral unless explicitly saved as notebook output (Save Version) or downloaded. **This archive may be large because it includes resumable checkpoints and the exact staged training images/metadata required for reliable resume. Excluding staged data makes a standalone resume archive incomplete.** Use `ARCHIVE_INCLUDE_CHECKPOINTS=False` if you only need diagnostics + final adapter.

In [ ]:
import zipfile
ARCHIVE_INCLUDE_CHECKPOINTS=True
ARCHIVE_INCLUDE_STAGED_DATA=True  # necessary when restoring optimizer checkpoints on a fresh Kaggle session
archive=Path('/kaggle/working')/(RUN_DIR.name+'_acne_lora_run.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=3) as z:
    for p in RUN_DIR.rglob('*'):
        if p.is_file():
            if not ARCHIVE_INCLUDE_CHECKPOINTS and 'checkpoint-' in str(p.relative_to(RUN_DIR)): continue
            if not ARCHIVE_INCLUDE_STAGED_DATA and 'captioned_acne_train' in p.parts: continue
            z.write(p,arcname=str(p.relative_to(RUN_DIR)))
print('Archive:',archive,'size MB:',round(archive.stat().st_size/1e6,1))
print('IMPORTANT: Use Kaggle Save Version to persist output, or download the archive.')

## Error troubleshooting checklist

1. **No GPU / CUDA OOM:** select T4 in Kaggle Settings. If OOM occurs, restart session to clear VRAM; confirm single process, batch size 1, gradient checkpointing, and close other pipelines. A 16 GB GPU is a starting requirement, not a guarantee.
2. **Repository cannot be cloned or model unavailable:** enable Internet, check Hugging Face availability/terms, and rerun the script/model steps. Do not silently switch checkpoints.
3. **`ModuleNotFoundError` or import mismatch:** restart session after pip installs, rerun version checks, capture full traceback and `environment.json`.
4. **Training interrupted:** restore the original run's `output/` folder with an `checkpoint-STEP` directory. Set `RUN_MODE="full"` and `RESUME_RUN_DIR` to that **output** directory; The exact original staged images, metadata, and trainer settings must remain available. This notebook checks these on resume. Do not change dataset ordering, steps or library versions mid-resume.
5. **No generated images or `pytorch_lora_weights.safetensors`:** final adapter is only exported when training completes; resume from most recent checkpoint.
6. **Skin lesions cropped out:** do not train the full run. Revisit data framing and caption quality; refrain from augmentations that distort disease morphology.
7. **Loss decreasing but poor clinical features:** this is plausible. Review held-out prompt performance, expert fidelity evaluation, and memorization tests before claiming success.
8. **Sharing logs:** provide `train.log`, `command.json`, `environment.json`, `gpu_usage.csv`, loss plot and sample grid; **do not publish identifiable patient images**.

**Storage note:** To resume from a downloaded archive on a new Kaggle session, restore the entire original run directory, not just the `output/` folder.

**Research limitation:** This is a baseline training-and-reproducibility notebook, not a validated medically accurate generator. Image-specific captions, licensing/privacy review, and a real validation protocol remain necessary before making strong scientific claims.
